<a href="https://colab.research.google.com/github/Ayesha-Zaman/Intelligence-Journey-/blob/main/07%3A%20Advanced%20Pandas/%20Topic%2020%3A%20Merging%2C%20Joining%20%26%20Concatenating/%20Task%2020%3A%20Merging%2C%20Joining%20%26%20Concatenating.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

DATA DESCRIPTION
```
file name -> Columns
quater-i.csv -> ['order_id', 'quantity', 'item_id', 'choice_description_id' 'item_price']
items.csv -> ['item_id', 'item_name']
```
Dataset Link - https://drive.google.com/drive/folders/1Z0kaFybvgFeczeUj4dldUnhTdloLqLsL?usp=share_link

In [2]:
# import like this
items_path = "/content/items.csv"
q1_path = "/content/quarter-1.csv"
q2_path = "/content/quarter-2.csv"
q3_path = "/content/quarter-3.csv"


q1= pd.read_csv(q1_path)
q2 = pd.read_csv(q2_path)
q3 = pd.read_csv(q3_path)

items = pd.read_csv(items_path)

###`Q:1-5`
1. You are given three quater files, your job is to append these three files and make a single dataframe.
2. Have a index as Q-1 Q-2 Q-3 for respective quater files in the dataframe
3. Your are given a file items.csv which has item_id and item_name. Find out most sold items in each quarter.
4. Find out items which has made most revenue in each quarter.
5. Find out avg order price of each quarter.

***Note: item_price is given as str with $ sign, in earlier task you have converted this to rupees, here too first convert item_price field in rupees.***

In [3]:
# 1+2
df = pd.concat([q1,q2,q3], keys=['Q1','Q2','Q3'])
df

order_id quantity item_id choice_description_id item_price
Q1 0           1        1       1                     1     $3.39 
   1           1        1       2                     2     $3.39 
   2           2        2       4                     3    $16.98 
   3           4        1       7                     6     $9.25 
   4           6        1       9                     8     $8.75 
...          ...      ...     ...                   ...        ...
Q2 2342     1829        1      23                    92    $11.25 
   2343     1830        1      23                  1043    $11.25 
   2344     1832        1      10                   116     $8.75 
   2345     1832        1       8                     0     $4.45 
   2346     1834        1      20                   515    $11.25 

[4622 rows x 5 columns]

In [4]:
# 3
new_df = df.reset_index().merge(items, on='item_id')
new_df.rename(columns={'level_0':'quater'}, inplace=True)
new_df.groupby(['quater','item_name'], as_index=False)['quantity']\
.sum().sort_values(by='quantity', ascending=False).drop_duplicates(subset='quater',keep='first')

,quater,item_name,quantity
65,Q2,Chicken Bowl,394
17,Q1,Chicken Bowl,367


In [5]:
# 4
new_df['item_price'] = new_df.item_price.apply(lambda x: float(x[1:]))
new_df['total_item_price'] = new_df['item_price'] * new_df['quantity']
new_df.groupby(['quater','item_name'], as_index=False)['total_item_price']\
.sum().sort_values(by='total_item_price', ascending=False).drop_duplicates(subset='quater',keep='first')

,quater,item_name,total_item_price
65,Q2,Chicken Bowl,4192.25
17,Q1,Chicken Bowl,3852.38


In [6]:
new_df.groupby(['quater','order_id'], as_index=False)['total_item_price'].sum().groupby('quater', as_index=False)['total_item_price'].mean()

,quater,total_item_price
0,Q1,13.809488
1,Q2,13.279828


###`Q-6` From the IPL dataset you have to find the Purple cap holder each season.

*Note: Bowler with most no wickets in a season gets purple cap. If more than one bowler have same no of wickets in the season, one with least ecomnomy among them is purple cap holder.*

Bowler's Economy = runs-conceded per six balls

In [7]:
balls = pd.read_csv("/content/IPL_Ball_by_Ball_2008_2022.csv")
matches = pd.read_csv("/content/IPL_Matches_2008_2022.csv")

In [8]:
# code
seasondf = balls.merge(matches[["ID", "Season"]], on="ID")
seasondf['IsBowlerWicket'] = seasondf.kind.apply(lambda x: 1 if x in ['caught', 'caught and bowled', 'bowled', 'stumped','lbw', 'hit wicket'] else 0)
seasondf['BowlerRun'] = seasondf.extra_type.apply(lambda x: 0 if x in ['legbyes', 'byes'] else 1)
seasondf['IsLegalBall'] = seasondf.extra_type.apply(lambda x: 0 if x in ["wides", "noballs"] else 1)
pcap_df = seasondf.groupby(["Season", "bowler"] ,as_index=False)[['IsBowlerWicket','BowlerRun','IsLegalBall']].sum()
pcap_df['Economy'] = pcap_df["BowlerRun"] / pcap_df["IsLegalBall"] * 6
pcap_df.sort_values(['IsBowlerWicket','Economy'], ascending=[False,True]).drop_duplicates("Season", keep = "first").sort_values(by='Season')

,Season,bowler,IsBowlerWicket,BowlerRun,IsLegalBall,Economy
41,2020/21,K Rabada,32,406,398,6.120603
127,2021,HV Patel,32,359,338,6.372781
331,2022,YS Chahal,27,420,408,6.176471


Match Result from- https://www.mykhel.com/cricket/ipl-purple-cap-winners-list-s4/

### `Q-7:` Best bowler in death overs.
*Note: Have taken most no of wickets in case of tie with least economy*

Death Overs - [16-20]

In [9]:
death_overs = seasondf[seasondf.overs >= 15]
pcap_df = death_overs.groupby("bowler" ,as_index=False)[['IsBowlerWicket','BowlerRun','IsLegalBall']].sum()
pcap_df['Economy'] = pcap_df["BowlerRun"] / pcap_df["IsLegalBall"] * 6
pcap_df.sort_values(['IsBowlerWicket','Economy'], ascending=[False,True]).head()

,bowler,IsBowlerWicket,BowlerRun,IsLegalBall,Economy
57,K Rabada,36,332,322,6.186335
52,JJ Bumrah,32,425,416,6.129808
40,HV Patel,32,338,317,6.397476
84,Mohammed Shami,31,282,278,6.086331
54,JO Holder,26,217,203,6.413793
